# ch02 核心指标计算与可视化

计算对照组和实验组的点击率及置信区间，并可视化对比。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Config
from src.utils.data_loader import DataLoader
from src.utils.visualizer import Visualizer
from src.utils.output_manager import OutputManager

print("模块加载成功")

## Step 1: 加载清洗后数据

In [ ]:
config = Config()
loader = DataLoader(config)
visualizer = Visualizer(config)
output = OutputManager(config)

df = loader.load_processed("cleaned_data.csv")
print(f"数据形状: {df.shape}")
df.head()

## Step 2: 计算分组核心指标

In [ ]:
# 分组统计
group_stats = df.groupby('group').agg({
    'user_id': 'count',
    'click': ['sum', 'mean']
})
group_stats.columns = ['n', 'clicks', 'ctr']
group_stats = group_stats.reset_index()

# 计算提升
ctr_con = group_stats[group_stats['group'] == 'con']['ctr'].values[0]
ctr_exp = group_stats[group_stats['group'] == 'exp']['ctr'].values[0]

absolute_lift = ctr_exp - ctr_con
relative_lift = (ctr_exp - ctr_con) / ctr_con * 100

print(f"对照组 CTR: {ctr_con:.4f}")
print(f"实验组 CTR: {ctr_exp:.4f}")
print(f"绝对提升: {absolute_lift:.4f}")
print(f"相对提升: {relative_lift:.2f}%")

output.save_table(group_stats, "group_metrics", chapter_prefix="ch02")

## Step 3: 计算 95% 置信区间

In [ ]:
# 计算标准误和置信区间
def calc_ci(p, n, z=1.96):
    se = np.sqrt(p * (1 - p) / n)
    ci_lower = p - z * se
    ci_upper = p + z * se
    return se, ci_lower, ci_upper

results = []
for _, row in group_stats.iterrows():
    se, ci_lower, ci_upper = calc_ci(row['ctr'], row['n'])
    results.append({
        'group': row['group'],
        'n': row['n'],
        'clicks': row['clicks'],
        'ctr': row['ctr'],
        'se': se,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper
    })

metrics_with_ci = pd.DataFrame(results)
print(metrics_with_ci)
output.save_table(metrics_with_ci, "group_metrics_with_ci", chapter_prefix="ch02")

## Step 4: 绘制可视化图表

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

groups = metrics_with_ci['group'].tolist()
ctrs = metrics_with_ci['ctr'].tolist()
ci_lowers = metrics_with_ci['ci_lower'].tolist()
ci_uppers = metrics_with_ci['ci_upper'].tolist()

errors = [[c - l for c, l in zip(ctrs, ci_lowers)],
          [u - c for c, u in zip(ctrs, ci_uppers)]]

colors = ['#DD8452', '#4C72B0']
bars = ax.bar(groups, ctrs, yerr=errors, capsize=10, color=colors)

# 添加数值标签
for bar, ctr in zip(bars, ctrs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{ctr:.2%}', ha='center', va='bottom', fontsize=12)

ax.set_title('A/B测试点击率对比（含95%置信区间）', fontsize=14)
ax.set_xlabel('分组', fontsize=12)
ax.set_ylabel('点击率', fontsize=12)
ax.set_ylim(0, max(ci_uppers) * 1.2)
ax.legend(['对照组 (con)', '实验组 (exp)'])

plt.tight_layout()
visualizer.save_figure(fig, "ctr_comparison_with_ci", chapter_prefix="ch02")
plt.show()